In [ ]:
import json

import hardware
import matplotlib.pyplot as plt
import numpy as np
import torch

hw = hardware.load("2026-11-07-hd1910")   # 训练用的硬件版本
hw.servo.actuator                         # 舵机：参数文件与固件刚度

In [ ]:
params = json.loads((hw.servo.dir / hw.servo.actuator["params_json"]).read_text())
kt, R, eta = params["kt"], params["R"], params["error_gain_ratio"]   # 转矩常数、线圈电阻、换算修正
kt, R, eta

In [ ]:
GAIN, D_MAX = 0.166, 0.97                                # 飞特固件：误差换算成占空比的系数、占空比上限


def torque(err, omega, vin=6.0, kp=5.0):
    d = np.clip(GAIN * eta * kp * err, -D_MAX, D_MAX)    # 固件：角度误差 → 占空比
    i = (d * vin - kt * omega) / R                       # 电机：电压减反电动势 → 电流
    return kt * i                                        # 减速箱：电流 → 力矩（不计摩擦）


torque(np.radians(5), 0.0)                               # 误差 5°、关节静止、6 伏

In [ ]:
omega = np.linspace(0, 10, 101)                          # 关节转速 0–10 rad/s
for vin in (6.0, 5.0):
    plt.plot(omega, np.maximum(torque(np.pi, omega, vin), 0), label=f"{vin} V")   # 误差很大：占空比用满
plt.xlabel("rad/s")
plt.ylabel("N·m")
plt.legend()

In [ ]:
torque(np.radians(5), 0.0, kp=32)                        # 刚度换成出厂值 32

In [ ]:
from env import SKILLS, DuckEnv
from rollout import latest_checkpoint, load_checkpoint

policy, ckpt = load_checkpoint(latest_checkpoint("m04-walk"), "cuda")   # 上一讲训的走路策略，取最新一份
env = DuckEnv(SKILLS["walk"], hw, num_envs=64, device="cuda")
env.set_curriculum_step(ckpt["common_step_counter"])                     # 课程表对齐到训练到的那一档
robot = env.unwrapped.scene["robot"]

In [ ]:
env.reset()
env.pin_commands(0.3)                                    # 一直往前走 0.3 m/s
obs, _ = env.get_observations()
knee = robot.joint_names.index("right_knee")
target, actual = [], []
with torch.no_grad():
    for _ in range(150):                                 # 3 秒
        obs, _, _, _, _ = env.step(policy.act_inference(obs))
        target.append(robot.data.joint_pos_target[0, knee].item())
        actual.append(robot.data.joint_pos[0, knee].item())
plt.plot(np.degrees(target), "--", label="target")
plt.plot(np.degrees(actual), label="actual")
plt.xlabel("step (20 ms)")
plt.ylabel("deg")
plt.legend()

In [ ]:
LAST_ACTION = slice(34, 48)                              # 观测里「上一拍动作」的位置


def turn(alpha, steps=300):
    env.reset()
    env.pin_commands(0.2, 0.0, 1.0)                      # 边走边左转，指令 1.0 rad/s
    obs, _ = env.get_observations()
    prev = torch.zeros(64, 14, device="cuda")            # 上一拍下发的目标
    raw_last = torch.zeros(64, 14, device="cuda")        # 上一拍策略的原始输出
    yaw = []
    with torch.no_grad():
        for _ in range(steps):
            obs[:, LAST_ACTION] = raw_last               # 观测里放策略的原始输出，和固件一样
            raw = policy.act_inference(obs)
            sent = alpha * raw + (1 - alpha) * prev      # 一阶低通：这一拍与上一拍加权
            obs, _, _, _, _ = env.step(sent)
            prev, raw_last = sent, raw
            yaw.append(robot.data.root_link_ang_vel_b[:, 2].mean().item())
    return np.mean(yaw[100:])                            # 去掉起步的 2 秒，平均转向角速度

In [ ]:
for alpha in (1.0, 0.95, 0.7, 0.5):
    print(alpha, round(turn(alpha), 3))